# 03 — Morphological & Thermal Parameters

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ByMaxAnjos/LCZ4py/blob/main/notebooks/general/03_morphological_parameters.en.ipynb)

**Learning objective**: go beyond an LCZ *class map* to the 34 quantitative morphological and thermal parameters that Stewart & Oke (2012) attach to every LCZ class — sky view factor, aspect ratio, surface fractions, roughness, admittance, albedo, and anthropogenic heat — and learn to extract, subset, vectorize, and visualize them with `lcz_get_parameters` and `lcz_plot_parameters`.

## Why parameters, not just classes

The Local Climate Zone scheme (Stewart & Oke, 2012, *Bulletin of the American Meteorological Society*) is often introduced as a *classification* — 17 discrete categories from "Compact highrise" (LCZ 1) to "Water" (LCZ 17). But the classification itself is only half of the framework's value. What makes LCZ genuinely useful for urban climate science is that **every class carries a standardized, literature-derived range of physical parameters** that control the surface energy balance and near-surface climate: how much sky a street canyon can see (sky view factor, SVF), how tall buildings are relative to the space between them (aspect ratio / height-to-width, roughly what the raw band `aspect_mean` encodes here), how much of the ground is built on versus paved versus vegetated (building/impervious/pervious/tree surface fractions — BSF/ISF/PSF/TSF), how rough the surface is aerodynamically (roughness class and `z0`, the roughness length), how quickly the surface material stores and releases heat (surface (thermal) admittance, SAD), how much shortwave radiation it reflects (surface albedo, SAL), and how much waste heat human activity injects directly into the urban boundary layer (anthropogenic heat flux, AH).

This is precisely the reason LCZ overtook the older "urban vs. rural" UHI paradigm: a "Compact midrise" zone in São Paulo and one in Berlin will have *systematically similar* radiative and aerodynamic properties even though they look nothing alike administratively, because the classification is built directly from the morphology, not from land-use labels. That portability is what makes LCZ maps usable as **direct inputs to micro- and meso-scale urban climate models** — the parameter ranges here are exactly the kind of lookup table that WRF's urban canopy models (single-layer or multi-layer UCM, BEP/BEM) and other mesoscale/microscale schemes need to initialize morphology-dependent physics without site-specific building surveys. A modeler who has an LCZ map for a region effectively already has first-guess values for surface roughness, thermal admittance, and albedo — the exact fields this notebook extracts.

`LCZ4py` implements this parameter table as a fast NumPy lookup (see `lcz_get_parameters`), following the same conceptual design as the R package `LCZ4r` (Anjos, M. et al., 2025, *Scientific Reports*, https://www.nature.com/articles/s41598-025-92000-0), reimplemented here with vectorized indexing that is 30–100x faster than the original `dplyr`-join approach. Each of the 17 classes maps to a `min`/`max`/`mean` triplet for 11 parameter families (SVF, AR, BSF, ISF, PSF, TSF, HRE — height of roughness elements, TRC — terrain roughness class, SAD, SAL, AH) plus a single roughness-length value (`z0`), for **34 real parameters** (plus the LCZ class code itself as a bookkeeping band) — pixel by pixel, driven entirely by the class map you already downloaded with `lcz_get_map`.

In [1]:
!pip -q install "LCZ4py[all] @ git+https://github.com/ByMaxAnjos/LCZ4py.git"


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: /Applications/QGIS.app/Contents/MacOS/bin/python3 -m pip install --upgrade pip
ERROR: Package 'lcz4py' requires a different Python: 3.9.5 not in '>=3.10'


## 1. Get an LCZ class map

We use **Vaduz** (Liechtenstein) as our reference city throughout — small enough that the map, parameter extraction, and every chart below run in seconds, and diverse enough (it contains several urban and natural/water classes) to make the comparisons meaningful.

In [2]:
from LCZ4py import lcz_get_map

map_path = lcz_get_map(city="Vaduz")
print(map_path)

06:26:12 - LCZ4py._internal._lcz_map_engine - INFO - Loading clipped map from local cache.


/Users/co2map/.lcz4r_cache/clipped_a8db9faa23e32c3b.tif


## 2. Extract the full 34-parameter stack — `lcz_get_parameters`

`lcz_get_parameters(x, istack=True)` maps the LCZ class raster to a `(35, H, W)` array in one vectorized NumPy pass: band 0 is the LCZ class code itself (kept for bookkeeping/joins), and the remaining 34 bands are the `min`, `max`, and `mean` variants of the parameter families described above, plus roughness length `z0`. With `isave=True` it also writes a multi-band GeoTIFF (`LCZ4r_output/lcz_par_stack.tif`) with each band's name stored as its GDAL band description — this is what lets `lcz_plot_parameters` (and, later, `lcz_cal_indices` under the hood) address bands by name instead of position, and it's the file we'll reuse for every chart below.

Returns an `LCZStackResult` with `.path` (if saved), `.array` (the in-memory stack), `.gdf`/`.geoarrow_table` (unused for this call — those are populated by the `ishp=True` path below).

In [3]:
from LCZ4py import lcz_get_parameters

stack_result = lcz_get_parameters(map_path, istack=True, isave=True)
print("Stack shape (bands, height, width):", stack_result.array.shape)
print("Saved GeoTIFF:", stack_result.path)

06:26:12 - rasterio._env - WARNING - CPLE_IllegalArg in lcz_par_stack.tif: BLOCKXSIZE can only be used with TILED=YES


06:26:12 - LCZ4py.general.lcz_get_parameters - INFO - Saved: /Users/co2map/Documents/lcz4r_python/notebooks/general/LCZ4r_output/lcz_par_stack.tif


Stack shape (bands, height, width): (35, 120, 131)
Saved GeoTIFF: LCZ4r_output/lcz_par_stack.tif


**Reading the output**: the shape `(35, H, W)` confirms one band per class-code + 34 parameters, at the same pixel grid as the input LCZ map. Every pixel's value is looked up directly from its LCZ class — this is a categorical-to-continuous remap, not an independent measurement, so all pixels sharing an LCZ class get identical parameter values (the *within-class range* between `min` and `max` reflects real-world variability across cities from the Stewart & Oke literature synthesis, not variability within this specific raster).

## 3. Select a parameter subset — `iselect`

Most workflows don't need all 34 bands. Passing `iselect` (a name or list of names) returns just those bands as a plain NumPy array — useful when you only need, say, the mean surface fractions and anthropogenic heat as model-ready rasters, without paying the I/O cost of the full stack.

In [4]:
selected = lcz_get_parameters(
    map_path, iselect=["svf_mean", "BSF_mean", "AH_mean"]
)
print("Selected shape:", selected.shape, type(selected))

Selected shape: (3, 120, 131) <class 'numpy.ndarray'>


**Reading the output**: shape `(3, H, W)` — one band per requested name, in request order — `svf_mean` (mean sky view factor), `BSF_mean` (mean building surface fraction), `AH_mean` (mean anthropogenic heat flux, W/m²). This is the leanest way to pull exactly the fields a downstream model or analysis needs.

## 4. Vectorize to a polygon table — `ishp=True`

`ishp=True` dissolves the class raster into one polygon per contiguous LCZ zone (via DuckDB Spatial when available, falling back to GeoPandas) and joins every parameter onto it as attribute columns. This is the natural format for GIS workflows — e.g. handing a per-zone parameter table to a WRF preprocessor, a QGIS layer, or a GeoPackage for a modeling team that doesn't work in Python/raster space.

In [5]:
gdf_result = lcz_get_parameters(map_path, ishp=True)
print(gdf_result.gdf.shape, "polygons x columns")
gdf_result.gdf[["lcz", "svf_mean", "BSF_mean", "AH_mean", "z0", "geometry"]].head()

06:26:12 - LCZ4py.general.lcz_get_parameters - INFO - Vectorizing LCZ classes...


06:26:12 - rasterio._env - WARNING - CPLE_AppDefined in DeprecationWarning: 'Memory' driver is deprecated since GDAL 3.11. Use 'MEM' onwards. Further messages of this type will be suppressed.


06:26:12 - LCZ4py.general.lcz_get_parameters - WARNING - DuckDB dissolve failed, falling back to GeoPandas: Not implemented Error: Data type 'geometry' not recognized


(9, 36) polygons x columns


,lcz,svf_mean,BSF_mean,AH_mean,z0,geometry
0,5,0.65,30.0,24.0,0.50,"POLYGON ((9.52124 47.14359, 9.52124 47.14269, ..."
1,6,0.75,30.0,24.0,0.50,"POLYGON ((9.50418 47.15706, 9.50418 47.15616, ..."
2,8,0.75,40.0,49.0,0.25,"MULTIPOLYGON (((9.51765 47.12292, 9.51765 47.1..."
3,9,0.85,15.0,9.0,0.50,"MULTIPOLYGON (((9.51855 47.13191, 9.51855 47.1..."
4,11,0.35,9.0,0.0,2.00,"MULTIPOLYGON (((9.60928 47.09867, 9.60928 47.0..."


**Reading the output**: one row per LCZ zone present in Vaduz's footprint, with the full parameter table joined by class (`lcz`) plus a `geometry` column. Note that `z0` and every `min`/`max`/`mean` field are *class constants* here, not per-polygon estimates — the polygon geometry is what varies (its shape and area), the attributes are the literature-derived lookup values for that class.

## 5. Visualize spatial patterns — `chart_type="map"`

`lcz_plot_parameters` renders any parameter band as an interactive Plotly heatmap. We use the saved stack path from step 2 (which carries band names as GDAL descriptions) and `iselect` three representative mean parameters: mean sky view factor (`svf_mean`), mean building surface fraction (`BSF_mean`), and mean anthropogenic heat (`AH_mean`).

In [6]:
from LCZ4py import lcz_plot_parameters

map_figs = lcz_plot_parameters(
    stack_result.path, iselect=["svf_mean", "BSF_mean", "AH_mean"],
    chart_type="map",
)
for fig in map_figs:
    fig.show()

**Reading the output**: three heatmaps over the exact footprint of the LCZ map. `svf_mean` should be *lowest* in the compact/dense classes (narrow street canyons see little sky) and highest over open/natural land; `BSF_mean` and `AH_mean` should track each other and peak wherever the map's densest built classes sit (industry/compact zones), and drop to ~0 over vegetation and water. If the three maps don't show this inverse SVF-vs-BSF/AH pattern, that's a sign something's off with the input class raster, not with the parameter table.

## 6. Parameter-vs-parameter relationships — `chart_type="scatter"`

For `chart_type` values other than `"map"`, `lcz_plot_parameters` delegates to `lcz_cal_indices` (the same per-class statistics engine used for spectral indices) — it works on *any* multi-band GeoTIFF with named bands, morphological parameters included, since the aggregation logic never assumes anything about what the bands physically represent. This requires `lcz_map` (the class raster, to assign each pixel's LCZ class for coloring/statistics).

`"scatter"` plots every pixel's `index_x` against `index_y`, colored by LCZ class using the official LCZ palette — here, mean building surface fraction vs. mean sky view factor.

In [7]:
scatter_fig = lcz_plot_parameters(
    stack_result.path, lcz_map=map_path, chart_type="scatter",
    index_x="BSF_mean", index_y="svf_mean",
)
scatter_fig.show()

**Reading the output**: because both axes are class *constants* (not per-pixel measurements), each LCZ class collapses to a single point (or a small cluster where the raster is a mix of nearby resolutions/edge pixels) rather than a spread cloud — which itself is the point: it makes the built-in Stewart & Oke inverse relationship between building density and open sky explicit and class-by-class comparable. Dense/compact classes sit in the low-SVF/high-BSF corner; open, natural, and water classes sit in the opposite corner.

## 7. Per-class profile shape — `chart_type="radar"`

`"radar"` draws one polar trace per LCZ class across several requested parameters, each axis min-max normalized (`normalize_radar=True`, the default) so parameters with very different natural ranges (SVF's 0–1 vs. AH's tens-to-hundreds of W/m²) stay visually comparable on the same chart. This is the clearest way to compare the *overall morphological signature* of one class against another — not one parameter at a time, but the whole shape.

In [8]:
profile_params = [
    "svf_mean", "aspect_mean", "BSF_mean", "ISF_mean",
    "PSF_mean", "TSF_mean", "AH_mean",
]

radar_fig = lcz_plot_parameters(
    stack_result.path, iselect=profile_params, lcz_map=map_path,
    chart_type="radar",
)
radar_fig.show()

**Reading the output**: `aspect_mean` here is the mean height/width aspect ratio band (labelled `aspect_mean` in the raster, corresponding to Stewart & Oke's "AR"). Compact/dense urban classes should trace a profile skewed toward high `BSF_mean`/`ISF_mean`/`aspect_mean`/`AH_mean` and low `svf_mean`/`TSF_mean`/`PSF_mean`; natural classes trace close to the mirror image. A class whose polygon looks "spiky" in one axis relative to its neighbors is the one with the most distinctive morphology for that parameter — useful for spotting which classes a given urban intervention (e.g. adding tree cover) would move most.

## 8. Redundancy among parameters — `chart_type="correlation"`

`"correlation"` computes the Pearson correlation matrix between the requested parameters over every valid LCZ pixel. With 34 parameters built from only 11 base families (each split into min/max/mean), a lot of redundancy is expected — this chart makes it explicit, which matters if you're about to feed these fields into a statistical or ML model where near-duplicate predictors just add collinearity without adding information.

In [9]:
corr_fig = lcz_plot_parameters(
    stack_result.path, iselect=profile_params, lcz_map=map_path,
    chart_type="correlation",
)
corr_fig.show()

**Reading the output**: `BSF_mean` and `ISF_mean` (building and impervious surface fraction) typically show a strong positive correlation — both measure "how paved-over is this class," just from slightly different angles — while `svf_mean` typically anti-correlates strongly with `aspect_mean` and `BSF_mean` (more building volume blocks more sky, by construction). `TSF_mean`/`PSF_mean` (tree/pervious fraction) tend to anti-correlate with the built-up trio. If you were selecting a minimal feature set for, say, a machine-learning interpolation model (see `lcz_interp_map_plus` in the `local/` series), this matrix is exactly how you'd decide which parameters are redundant and safe to drop.

## Recap & next steps

In this notebook you moved from an LCZ *class* map to its full **34-parameter morphological/thermal signature** using `lcz_get_parameters` — extracting the full stack, a lean subset (`iselect`), and a GIS-ready polygon table (`ishp=True`) — and then explored that signature four ways with `lcz_plot_parameters`: spatial pattern (`"map"`), pairwise relationships (`"scatter"`), per-class profile shape (`"radar"`), and cross-parameter redundancy (`"correlation"`).

- Previous: [`02_visualization_area_stats`](02_visualization_area_stats.en.ipynb) — visualization and area statistics.
- Next: [`04_remote_sensing_lst_pc`](04_remote_sensing_lst_pc.en.ipynb) — pulling in real remote-sensing observations (Land Surface Temperature, Microsoft Planetary Computer) to compare against these literature-derived parameter estimates.